# XM Price Checker

Notebook controller for reading the Excel configuration, running the enabled website scrapers, collecting variant-level price/availability data, and writing the results to `RawData`.

The actual website scraping remains in `scrapers/` and is executed through `run_price_check.py`.


In [26]:
# ============================================================
# 1. SETUP
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys

from openpyxl import load_workbook
import pandas as pd

project_folder = Path(
    r"C:\Users\New\.vscode\Python_projects\XM_Price_Checker"
)

excel_file = project_folder / "XM_Price_Checker_Python.xlsm"

print("Project folder:", project_folder)
print("Excel exists:", excel_file.exists())


Project folder: C:\Users\New\.vscode\Python_projects\XM_Price_Checker
Excel exists: True


In [27]:
# ============================================================
# 2. LOAD EXCEL WORKBOOK
# ============================================================

workbook = load_workbook(
    excel_file,
    keep_vba=True
)

products_sheet = workbook["Products"]
links_sheet = workbook["Links"]
rawdata_sheet = workbook["RawData"]
results_sheet = workbook["Results"]

print("Workbook loaded successfully.")
print("Sheets:", workbook.sheetnames)
print("Products rows:", products_sheet.max_row)
print("Links rows:", links_sheet.max_row)
print("RawData rows:", rawdata_sheet.max_row)
print("Results rows:", results_sheet.max_row)


Workbook loaded successfully.
Sheets: ['Products', 'Links', 'RawData', 'Results', 'Settings']
Products rows: 30
Links rows: 125
RawData rows: 1
Results rows: 30


In [28]:
# ============================================================
# 3. LOAD PRODUCTS
# ============================================================

products = {}

for row in products_sheet.iter_rows(
    min_row=2,
    values_only=True
):
    if not row:
        continue

    model = row[1]
    name = row[2]
    ram = row[3]
    storage = row[4]

    if model is None or ram is None or storage is None:
        continue

    model = str(model).strip()
    ram = str(ram).strip()
    storage = str(storage).strip()

    target_id = f"{model}-{ram}-{storage}"

    products[target_id.upper()] = {
        "target_id": target_id,
        "name": str(name).strip() if name is not None else model,
        "model": model,
        "ram": ram,
        "storage": storage
    }

print("Products loaded:", len(products))

if products:
    print("Example:", next(iter(products.values())))


Products loaded: 28
Example: {'target_id': 'SOMALIAA-4-64', 'name': 'Redmi A7 pro', 'model': 'SOMALIAA', 'ram': '4', 'storage': '64'}


In [29]:
# ============================================================
# 4. LOAD ENABLED LINKS
# ============================================================

links = []

for row in links_sheet.iter_rows(
    min_row=2,
    values_only=True
):
    if not row:
        continue

    target_id = row[0]
    website = row[1]
    url = row[2]
    enabled = row[3]

    if (
        target_id
        and website
        and url
        and str(enabled).strip().upper() == "YES"
    ):
        links.append({
            "target_id": str(target_id).strip(),
            "website": str(website).strip().upper(),
            "url": str(url).strip()
        })

target_ids = sorted({
    item["target_id"].strip().upper()
    for item in links
})

print("Enabled links:", len(links))
print("Target IDs:", target_ids)


Enabled links: 9
Target IDs: ['LEEDSA-4-128', 'LEEDSA-4-256', 'P16-8-256', 'P16U-12-512', 'P16U-8-256', 'P17-6-128', 'P6-8-256', 'P7E-8-256', 'SOMALIAA-4-64']


In [30]:
# ============================================================
# 5. RUN PRICE CHECKER
# ============================================================

def run_price_check(website, url, product):
    """Run the website scraper in a separate Python process."""

    script = project_folder / "run_price_check.py"

    product_json = json.dumps(
        product,
        ensure_ascii=False
    )

    process = subprocess.run(
        [
            sys.executable,
            str(script),
            website,
            url,
            product_json
        ],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )

    # Keep scraper/browser logs visible in the notebook.
    print(process.stdout)

    if process.returncode != 0:
        print(process.stderr)
        raise RuntimeError(
            f"{website} price check failed."
        )

    for line in process.stdout.splitlines():
        if line.startswith("RESULT_JSON:"):
            return json.loads(
                line.replace(
                    "RESULT_JSON:",
                    "",
                    1
                ).strip()
            )

    raise ValueError(
        f"No RESULT_JSON returned for {website}."
    )


In [31]:
# ============================================================
# 6. PRODUCT LOOKUP
# ============================================================

def get_product(target_id):
    """Return product information loaded from the Products sheet."""

    key = str(target_id).strip().upper()

    try:
        return products[key]
    except KeyError:
        raise ValueError(
            f"TargetID not found in Products: {target_id}"
        )


In [32]:
# ============================================================
# 7. PRICE COLLECTION
# ============================================================

all_results = []

for target_id in target_ids:

    product = get_product(target_id)

    print()
    print("=" * 60)
    print("TARGET:", target_id)
    print(
        "Product:",
        product["name"],
        "| RAM:", product["ram"],
        "| Storage:", product["storage"]
    )
    print("=" * 60)

    target_links = [
        item for item in links
        if item["target_id"].strip().upper() == target_id
    ]

    for item in target_links:

        website = item["website"]
        url = item["url"]

        print()
        print("-" * 50)
        print("Website:", website)
        print("URL:", url)

        try:
            results = run_price_check(
                website,
                url,
                product
            )

            print("SCRAPER RESULTS:", results)

            # No matching product is not a scraper error.
            if not results:
                print("No matching products found.")
                continue

            for result in results:
                all_results.append({
                    "run_time": datetime.now(),
                    "target_id": target_id,
                    "website": website,
                    "product_name": result.get(
                        "product_name",
                        product["name"]
                    ),
                    "variant": result.get("variant", ""),
                    "ram": result.get("ram", product["ram"]),
                    "storage": result.get("storage", product["storage"]),
                    "url": url,
                    "price": result.get("price"),
                    "currency": "PLN",
                    "availability": result.get(
                        "availability",
                        "Unknown"
                    ),
                    "status": "OK",
                    "error": ""
                })

        except Exception as e:
            print("ERROR:", str(e))

            all_results.append({
                "run_time": datetime.now(),
                "target_id": target_id,
                "website": website,
                "product_name": product["name"],
                "variant": "",
                "ram": product["ram"],
                "storage": product["storage"],
                "url": url,
                "price": None,
                "currency": "PLN",
                "availability": "Unknown",
                "status": "ERROR",
                "error": str(e)
            })

print()
print("=" * 60)
print("PRICE COLLECTION COMPLETE")
print("=" * 60)
print("Total results:", len(all_results))



TARGET: LEEDSA-4-128
Product: Redmi 17 | RAM: 4 | Storage: 128

--------------------------------------------------
Website: AVANS
URL: https://www.avans.pl/search?query[menu_item]=&query[querystring]=Redmi%2017
AVANS MODULE PATH: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\scrapers\avans.py
NEONET MODULE PATH: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\scrapers\neonet.py
Website: AVANS
URL: https://www.avans.pl/search?query[menu_item]=&query[querystring]=Redmi%2017
Product: {'target_id': 'LEEDSA-4-128', 'name': 'Redmi 17', 'model': 'LEEDSA', 'ram': '4', 'storage': '128'}
Opening: https://www.avans.pl/search?query[menu_item]=&query[querystring]=Redmi%2017
Page loaded.
Waiting for Avans products...
Total Avans H2: 26
PRODUCT NAME MATCH: YES

AVANS CARD 0
TITLE: Smartfon XIAOMI Redmi 17 4G 4/128GB 6.9" 120Hz Czarny

AVANS FULL CARD TEXT:
------------------------------------------------------------
Smartfon XIAOMI Redmi 17 4G 4/128GB 6.9" 120Hz Czarny
Karta informacyj

In [33]:
# ============================================================
# 8. RESULTS / VERIFICATION
# ============================================================

results_df = pd.DataFrame(all_results)

print("Raw scraper results:", len(results_df), "rows")

if not results_df.empty:
    display(
        results_df[
            [
                "target_id",
                "website",
                "product_name",
                "variant",
                "ram",
                "storage",
                "price",
                "availability",
                "status",
                "error"
            ]
        ]
    )

    print()
    print("=" * 70)
    print("TARGET / WEBSITE SUMMARY")
    print("=" * 70)

    summary_df = (
        results_df
        .groupby(
            ["target_id", "website"],
            dropna=False
        )
        .agg(
            variants=("variant", "count"),
            available=(
                "availability",
                lambda x: (x == "Available").sum()
            ),
            unavailable=(
                "availability",
                lambda x: (x == "Unavailable").sum()
            ),
            prices=(
                "price",
                lambda x: x.notna().sum()
            ),
            errors=(
                "status",
                lambda x: (x == "ERROR").sum()
            )
        )
        .reset_index()
    )

    display(summary_df)
else:
    print("No scraper results were collected.")


Raw scraper results: 27 rows


,target_id,website,product_name,variant,ram,storage,price,availability,status,error
0,LEEDSA-4-128,AVANS,Redmi 17,Black,4 GB,128 GB,899.00,Available,OK,
1,LEEDSA-4-128,AVANS,Redmi 17,Blue,4 GB,128 GB,799.00,Available,OK,
2,LEEDSA-4-128,AVANS,Redmi 17,Green,4 GB,128 GB,799.00,Available,OK,
3,LEEDSA-4-256,AVANS,Redmi 17,Green,4 GB,256 GB,899.00,Available,OK,
4,LEEDSA-4-256,AVANS,Redmi 17,Blue,4 GB,256 GB,899.00,Available,OK,
5,LEEDSA-4-256,AVANS,Redmi 17,Black,4 GB,256 GB,899.00,Available,OK,
6,P16-8-256,AVANS,Redmi Note 15 Pro 5G,Black,8 GB,256 GB,1399.99,Available,OK,
7,P16-8-256,AVANS,Redmi Note 15 Pro 5G,Titanium,8 GB,256 GB,1399.00,Available,OK,
8,P16-8-256,AVANS,Redmi Note 15 Pro 5G,Blue,8 GB,256 GB,NaN,Unavailable,OK,
9,P16U-12-512,AVANS,Redmi Note 15 Pro+ 5G,Black,12 GB,512 GB,1899.99,Available,OK,



TARGET / WEBSITE SUMMARY


,target_id,website,variants,available,unavailable,prices,errors
0,LEEDSA-4-128,AVANS,3,3,0,3,0
1,LEEDSA-4-256,AVANS,3,3,0,3,0
2,P16-8-256,AVANS,3,2,1,2,0
3,P16U-12-512,AVANS,3,3,0,3,0
4,P16U-8-256,AVANS,3,3,0,3,0
5,P17-6-128,AVANS,3,3,0,3,0
6,P6-8-256,AVANS,3,2,1,2,0
7,P7E-8-256,AVANS,3,2,1,2,0
8,SOMALIAA-4-64,AVANS,3,3,0,3,0


In [37]:
# ============================================================
# 9. RAW DATA OUTPUT
# ============================================================
#
# The RawData sheet is append-only.
# If you want a fresh run instead of keeping historical rows,
# clear the existing RawData rows manually before running this cell.
# ============================================================

for _, result in results_df.iterrows():

    price = result["price"]

    # Pandas NaN -> blank Excel cell
    if pd.isna(price):
        price = None

    rawdata_sheet.append([
        result["run_time"],
        result["target_id"],
        result["website"],
        result["product_name"],
        result["variant"],
        result["url"],
        price,
        result["currency"],
        result["availability"],
        result["status"],
        result["error"]
    ])

print("RawData rows written:", len(results_df))


RawData rows written: 27


In [38]:
# ============================================================
# 10. GENERATE RESULTS SHEET
# ============================================================
#
# The Results sheet is the final report.
#
# IMPORTANT:
# - Matching between Products and RawData uses TargetID.
# - RRP is displayed but is NOT used for the comparison.
# - Only "RRP after Promotion" is used as the benchmark.
# - Offer start/end dates are copied to the report but are NOT used.
#
# Report logic for each website:
#
# 1. If all returned variants are Unavailable -> "unavailable"
#
# 2. Ignore unavailable variants.
#    Only available variants with a valid price are used.
#
# 3. If all available colours have the same price:
#
#    a. If the price equals the promotion price:
#       -> "OK"
#
#    b. If the price is different from the promotion price:
#       -> "<price>"
#
# 4. If available colours have different prices:
#
#    - Colours with the promotion price -> "OK <colour>"
#    - Colours with another price -> "<price> <colour>"
#    - Colours with the same price are grouped together.
#
# ============================================================

# ------------------------------------------------------------
# 10.1 NORMALIZE NUMBER
# ------------------------------------------------------------

def normalize_number(value):
    """
    Return a numeric value as float, or None.
    """

    if value is None or pd.isna(value):
        return None

    try:

        return float(value)

    except (TypeError, ValueError):

        return None


# ------------------------------------------------------------
# 10.2 NORMALIZE REPORT PRICE
# ------------------------------------------------------------

def normalize_report_price(value):
    """
    Convert a price into an integer for Results reporting.

    Decimal values are ignored.

    Examples:
    1899.99 -> 1899
    599.99  -> 599
    399.00  -> 399
    """

    price = normalize_number(value)

    if price is None:

        return None

    return int(price)


# ------------------------------------------------------------
# 10.3 FORMAT PRICE
# ------------------------------------------------------------

def format_price(price):
    """
    Convert a report price into clean text.

    Examples:
    1899 -> "1899"
    599  -> "599"
    """

    if price is None:

        return ""

    return str(int(price))


# ------------------------------------------------------------
# 10.4 BUILD ONE WEBSITE REPORT VALUE
# ------------------------------------------------------------

def build_report_value(
    raw_rows,
    promotion_price
):
    """
    Build the value that will be written into one
    website cell in Results H:O.
    """

    # --------------------------------------------------------
    # No scraper record at all
    # --------------------------------------------------------

    if not raw_rows:

        return ""

    # --------------------------------------------------------
    # Normalize promotion price for reporting
    #
    # Decimal values are ignored.
    # --------------------------------------------------------

    promotion_price = normalize_report_price(
        promotion_price
    )

    # --------------------------------------------------------
    # Separate available and unavailable variants
    # --------------------------------------------------------

    available_rows = []

    unavailable_count = 0

    for row in raw_rows:

        availability = str(
            row.get("availability", "")
        ).strip().lower()

        price = normalize_report_price(
            row.get("price")
        )

        # ----------------------------------------------------
        # Explicitly unavailable
        # ----------------------------------------------------

        if availability == "unavailable":

            unavailable_count += 1

            continue

        # ----------------------------------------------------
        # Valid available product
        # ----------------------------------------------------

        if (
            availability == "available"
            and price is not None
        ):

            available_rows.append(row)

    # --------------------------------------------------------
    # ALL VARIANTS UNAVAILABLE
    # --------------------------------------------------------

    if (
        not available_rows
        and unavailable_count > 0
    ):

        return "unavailable"

    # --------------------------------------------------------
    # No usable scraper result
    # --------------------------------------------------------

    if not available_rows:

        return ""

    # --------------------------------------------------------
    # Collect all available report prices
    #
    # Decimal values are ignored.
    # --------------------------------------------------------

    all_prices = [
        normalize_report_price(
            row.get("price")
        )
        for row in available_rows
    ]

    # --------------------------------------------------------
    # Remove invalid prices
    # --------------------------------------------------------

    all_prices = [
        price
        for price in all_prices
        if price is not None
    ]

    if not all_prices:

        return ""

    unique_prices = set(
        all_prices
    )

    # --------------------------------------------------------
    # CASE 1 AND CASE 2:
    #
    # All available colours have the same price.
    # --------------------------------------------------------

    if len(unique_prices) == 1:

        common_price = all_prices[0]

        # ----------------------------------------------------
        # CASE 1:
        # Same price as promotion price -> OK
        # ----------------------------------------------------

        if (
            promotion_price is not None
            and common_price == promotion_price
        ):

            return "OK"

        # ----------------------------------------------------
        # CASE 2:
        # Same price, but different from promotion price
        # -> return the price only
        # ----------------------------------------------------

        return format_price(
            common_price
        )

    # --------------------------------------------------------
    # CASE 3:
    #
    # Available colours have different prices.
    #
    # Group colours by report price.
    # --------------------------------------------------------

    price_groups = {}

    for row in available_rows:

        price = normalize_report_price(
            row.get("price")
        )

        if price is None:

            continue

        color = str(
            row.get("variant", "")
        ).strip()

        if not color:

            color = "Unknown"

        # ----------------------------------------------------
        # Create price group if it does not exist
        # ----------------------------------------------------

        if price not in price_groups:

            price_groups[price] = []

        # ----------------------------------------------------
        # Avoid duplicate colour names
        # ----------------------------------------------------

        existing_colors = {
            existing.lower()
            for existing in price_groups[price]
        }

        if color.lower() not in existing_colors:

            price_groups[price].append(
                color
            )

    # --------------------------------------------------------
    # Build final report groups
    # --------------------------------------------------------

    report_parts = []

    for price, colors in price_groups.items():

        color_text = ",".join(
            color.lower()
            for color in colors
        )

        # ----------------------------------------------------
        # Colours at promotion price
        # ----------------------------------------------------

        if (
            promotion_price is not None
            and price == promotion_price
        ):

            report_parts.append(
                f"OK {color_text}"
            )

        # ----------------------------------------------------
        # Colours at another price
        # ----------------------------------------------------

        else:

            price_text = format_price(
                price
            )

            report_parts.append(
                f"{price_text} {color_text}"
            )

    # --------------------------------------------------------
    # Separate different price groups with semicolons
    # --------------------------------------------------------

    return "; ".join(
        report_parts
    )

# ------------------------------------------------------------
# 10.5 READ WEBSITE HEADERS FROM RESULTS
# ------------------------------------------------------------

website_columns = [
    results_sheet.cell(
        1,
        col
    ).value
    for col in range(
        8,
        16
    )
]

print(
    "Results website columns:"
)

print(
    website_columns
)


# ------------------------------------------------------------
# 10.6 CREATE RAWDATA LOOKUP
# ------------------------------------------------------------

raw_lookup = {}

for _, raw_row in results_df.iterrows():

    target_id = str(
        raw_row["target_id"]
    ).strip().upper()

    website = str(
        raw_row["website"]
    ).strip().upper()

    key = (
        target_id,
        website
    )

    raw_lookup.setdefault(
        key,
        []
    ).append(
        raw_row.to_dict()
    )


# ------------------------------------------------------------
# 10.7 WRITE WEBSITE RESULTS
# ------------------------------------------------------------

result_row_number = 2

for product_row in products_sheet.iter_rows(
    min_row=2,
    values_only=True
):

    if not product_row:
        continue

    # --------------------------------------------------------
    # CURRENT PRODUCTS STRUCTURE
    # --------------------------------------------------------

    model = product_row[1]

    ram = product_row[3]

    storage = product_row[4]

    # --------------------------------------------------------
    # Skip invalid product rows
    # --------------------------------------------------------

    if (
        model is None
        or ram is None
        or storage is None
    ):

        continue

    model = str(
        model
    ).strip()

    ram = str(
        ram
    ).strip()

    storage = str(
        storage
    ).strip()

    # --------------------------------------------------------
    # Promotion price comes from Results
    # --------------------------------------------------------

    promotion_price = results_sheet.cell(
        result_row_number,
        5
    ).value

    # --------------------------------------------------------
    # Construct TargetID
    # --------------------------------------------------------

    target_id = (
        f"{model}-{ram}-{storage}"
    ).upper()

    print()
    print(
        "=" * 60
    )

    print(
        f"Generating Results for: {target_id}"
    )

    print(
        f"Promotion price: {promotion_price}"
    )

    # --------------------------------------------------------
    # WRITE WEBSITE RESULTS H:O
    # --------------------------------------------------------

    for offset, website_column in enumerate(
        website_columns,
        start=8
    ):

        # ----------------------------------------------------
        # Empty website header
        # ----------------------------------------------------

        if not website_column:
            continue

        website_key = str(
            website_column
        ).strip().upper()

        # ----------------------------------------------------
        # Find scraper results
        # ----------------------------------------------------

        raw_rows = raw_lookup.get(
            (
                target_id,
                website_key
            ),
            []
        )

        # ----------------------------------------------------
        # Build final report value
        # ----------------------------------------------------

        report_value = build_report_value(
            raw_rows,
            promotion_price
        )

        # ----------------------------------------------------
        # WRITE ONLY H:O
        # ----------------------------------------------------

        results_sheet.cell(
            result_row_number,
            offset
        ).value = report_value

        print(
            f"  {website_column}: {report_value}"
        )

    # --------------------------------------------------------
    # Next product row
    # --------------------------------------------------------

    result_row_number += 1


# ------------------------------------------------------------
# 10.8 FINISHED
# ------------------------------------------------------------

print()
print(
    "=" * 70
)

print(
    "WEBSITE RESULTS GENERATED"
)

print(
    "=" * 70
)

print(
    "Rows processed:",
    result_row_number - 2
)

print(
    "Website columns written:",
    website_columns
)

print(
    "Columns A:G were NOT modified."
)

print(
    "Only columns H:O were written."
)


# ------------------------------------------------------------
# 10.9 PREVIEW
# ------------------------------------------------------------

report_preview = pd.DataFrame(
    results_sheet.values
)

display(
    report_preview
)

Results website columns:
['MEX', 'Electro.pl', 'Avans', 'MSH', 'KTR', 'XKOM', 'NEONET', 'Max electro']

Generating Results for: SOMALIAA-4-64
Promotion price: 399
  MEX: 
  Electro.pl: 
  Avans: OK
  MSH: 
  KTR: 
  XKOM: 
  NEONET: 
  Max electro: 

Generating Results for: LEEDSA-4-128
Promotion price: 799
  MEX: 
  Electro.pl: 
  Avans: 899 black; OK blue,green
  MSH: 
  KTR: 
  XKOM: 
  NEONET: 
  Max electro: 

Generating Results for: LEEDSA-4-256
Promotion price: 949
  MEX: 
  Electro.pl: 
  Avans: 899
  MSH: 
  KTR: 
  XKOM: 
  NEONET: 
  Max electro: 

Generating Results for: P16U-12-512
Promotion price: 1899
  MEX: 
  Electro.pl: 
  Avans: OK
  MSH: 
  KTR: 
  XKOM: 
  NEONET: 
  Max electro: 

Generating Results for: P16U-8-256
Promotion price: 1699
  MEX: 
  Electro.pl: 
  Avans: OK
  MSH: 
  KTR: 
  XKOM: 
  NEONET: 
  Max electro: 

Generating Results for: P16-8-256
Promotion price: 1399
  MEX: 
  Electro.pl: 
  Avans: OK
  MSH: 
  KTR: 
  XKOM: 
  NEONET: 
  Max electro: 


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,SKU,Memory,Product Name,RRP,RRP after Promotion,Offer start date,Offer end date,MEX,Electro.pl,Avans,MSH,KTR,XKOM,NEONET,Max electro
1,SOMALIAA,4+64,Redmi A7 pro,549,399,2026-08-17 00:00:00,2026-08-23 00:00:00,,,OK,,,,,
2,LEEDSA,4+128,Redmi 17,899,799,2026-08-31 00:00:00,2026-09-06 00:00:00,,,"899 black; OK blue,green",,,,,
3,LEEDSA,4+256,Redmi 17,1099,949,2026-08-31 00:00:00,2026-09-06 00:00:00,,,899,,,,,
4,P16U,12+512,Redmi Note 15 Pro+ 5G,2299,1899,2026-08-17 00:00:00,2026-08-23 00:00:00,,,OK,,,,,
5,P16U,8+256,Redmi Note 15 Pro+ 5G,1999,1699,2026-08-17 00:00:00,2026-08-23 00:00:00,,,OK,,,,,
6,P16,8+256,Redmi Note 15 Pro 5G,1699,1399,2026-08-17 00:00:00,2026-08-23 00:00:00,,,OK,,,,,
7,P17,6+128,Redmi Note 15 5G,1199,1199,2026-08-17 00:00:00,2026-08-23 00:00:00,,,"982 black,purple; 999 blue",,,,,
8,P6,8+256,Redmi Note 15 Pro,1499,1299,2026-08-17 00:00:00,2026-08-23 00:00:00,,,OK black; 1499 titanium,,,,,
9,P7E,8+256,Redmi Note 15,1099,999,2026-08-17 00:00:00,2026-08-23 00:00:00,,,1099 blue; OK black,,,,,


In [39]:
# ============================================================
# 11. SAVE WORKBOOK
# ============================================================

workbook.save(excel_file)

print("RawData and Results updated successfully.")
print("Workbook saved:", excel_file)


RawData and Results updated successfully.
Workbook saved: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\XM_Price_Checker_Python.xlsm
